# 🧬 AMR-Predictor-ML: Predicting Meropenem Resistance in *Klebsiella pneumoniae*
## Phase 2: Feature Engineering & Genomic Representation (K-mer Counting)

### 📋 Phase Overview
In this phase, we transition from raw genomic sequences to numerical representations suitable for Machine Learning models. Since sequence lengths vary and standard ML algorithms require fixed-size numerical matrices, we will implement an alignment-free **K-mer counting pipeline**.

### 🎯 Notebook Objectives
1. Understand the mathematical representation of DNA sequences via K-mers.
2. Develop a scalable and memory-efficient Python function to parse FASTA files and count K-mer frequencies.
3. Prototype the feature extraction pipeline on a small subset of genomes before scaling.
4. Define the matrix vocabulary size ($4^K$) and analyze memory requirements for the dataset.

In [1]:
import os
from collections import Counter

# 1. Path configuration
FASTA_DIR = "../data/raw/fasta"

def extract_kmers_from_fasta(fasta_path, k=6):
    """
    Parses a FASTA file, merges all contigs into a single sequence,
    and extracts K-mer frequency counts.
    """
    sequence_parts = []
    
    # Read the file and parse contigs
    with open(fasta_path, 'r') as file:
        for line in file:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                # Skip header lines (Contig IDs)
                continue
            # Capitalize to avoid case-sensitivity issues (A vs a)
            sequence_parts.append(line.upper())
            
    # Combine all contigs into one master sequence string
    full_sequence = "".join(sequence_parts)
    
    # Slide the window of size K across the sequence
    kmers = []
    for i in range(len(full_sequence) - k + 1):
        kmer = full_sequence[i:i+k]
        # Quality control: ensure no ambiguous bases (like 'N' or 'R') are included
        if 'N' not in kmer and 'R' not in kmer:
            kmers.append(kmer)
            
    # Count frequencies of each unique K-mer
    kmer_counts = Counter(kmers)
    return kmer_counts, len(full_sequence)

# 2. Test the function on one downloaded FASTA file
# Let's list the directory and pick the first available file
available_files = [f for f in os.listdir(FASTA_DIR) if f.endswith('.fasta')]

if len(available_files) > 0:
    test_file = available_files[0]
    test_path = os.path.join(FASTA_DIR, test_file)
    
    print(f"🔬 Testing K-mer extraction on file: {test_file}")
    counts, seq_len = extract_kmers_from_fasta(test_path, k=6)
    
    print(f"📏 Total Sequence Length: {seq_len:,} base pairs")
    print(f"📊 Number of Unique 6-mers Found: {len(counts):,}")
    print("\n🔝 Top 5 most frequent 6-mers:")
    for kmer, count in counts.most_common(5):
        print(f"  {kmer} ---> {count} times")
else:
    print("❌ No FASTA files found in data/raw/fasta/! Please double-check your path.")

🔬 Testing K-mer extraction on file: 573.67298.fasta
📏 Total Sequence Length: 844,511 base pairs
📊 Number of Unique 6-mers Found: 4,096

🔝 Top 5 most frequent 6-mers:
  GCGGCG ---> 1457 times
  GCTGGC ---> 1391 times
  GGCGGC ---> 1383 times
  GCCAGC ---> 1373 times
  GCGCTG ---> 1373 times


### 📊 Scaling K-mer Extraction to the Entire Dataset

#### 🎯 Objective
Apply the verified 6-mer extraction function to all 2,835 successfully downloaded *Klebsiella pneumoniae* FASTA files. This will construct our final numerical feature matrix ($X$), where each row represents a bacterial strain and each of the 4,096 columns represents a specific 6-mer frequency count.

#### 💾 Storage Note
The resulting dataframe will be merged with our AMR labels and saved as a dense matrix in `data/processed/` for the upcoming Machine Learning model training phase.

In [3]:
import os
import pandas as pd
from collections import Counter
from tqdm import tqdm

# 1. Paths configuration
FASTA_DIR = "../data/raw/fasta"
PROCESSED_DIR = "../data/processed"

# Define strict standard DNA alphabet
STANDARD_BASES = set("ATCG")

def extract_strict_kmers(fasta_path, k=6):
    """
    Parses a FASTA file and extracts ONLY strict canonical K-mers (composed solely of A, T, C, G).
    Ignores any IUPAC ambiguous degenerate bases (N, R, M, S, W, K, Y, B, D, H, V).
    """
    sequence_parts = []
    with open(fasta_path, 'r') as file:
        for line in file:
            line = line.strip()
            if not line or line.startswith('>'):
                continue
            sequence_parts.append(line.upper())
            
    full_sequence = "".join(sequence_parts)
    kmers = []
    
    for i in range(len(full_sequence) - k + 1):
        kmer = full_sequence[i:i+k]
        # Strict validation: Check if ALL characters in the K-mer belong to standard ACGT
        if set(kmer).issubset(STANDARD_BASES):
            kmers.append(kmer)
            
    return Counter(kmers)

# 2. Process all downloaded FASTA files
fasta_files = [f for f in os.listdir(FASTA_DIR) if f.endswith('.fasta')]
all_genome_features = []

print(f"⏳ Processing {len(fasta_files)} genomes with Strict ACGT Filter...")
for file_name in tqdm(fasta_files, desc="Strict K-mer Extraction"):
    genome_id = file_name.replace(".fasta", "")
    fasta_path = os.path.join(FASTA_DIR, file_name)
    
    try:
        kmer_counts = extract_strict_kmers(fasta_path, k=6)
        feature_dict = {"Genome ID": float(genome_id)}
        feature_dict.update(kmer_counts)
        all_genome_features.append(feature_dict)
    except Exception as e:
        print(f"❌ Error processing {file_name}: {str(e)}")

# 3. Build Feature Matrix
print("📊 Building Feature Matrix...")
df_features = pd.DataFrame.from_records(all_genome_features)
df_features.fillna(0, inplace=True)

# 4. Load Metadata and Deduplicate to prevent row inflation
metadata_path = os.path.join(PROCESSED_DIR, "cleaned_meropenem_metadata.csv")
df_metadata = pd.read_csv(metadata_path)
df_metadata['Genome ID'] = df_metadata['Genome ID'].astype(float)

# Deduplicate metadata based on Primary Key (Genome ID)
df_metadata_clean = df_metadata.drop_duplicates(subset=['Genome ID']).copy()
print(f"🧹 Metadata cleaned. Rows reduced from {len(df_metadata)} to {len(df_metadata_clean)} unique strains.")

# 5. Perfect Inner Merge
print("🔗 Performing strict merging...")
df_final_dataset = pd.merge(df_metadata_clean, df_features, on="Genome ID", how="inner")

# 6. Save the clean golden dataset
output_matrix_path = os.path.join(PROCESSED_DIR, "kmers_6_dataset.csv")
df_final_dataset.to_csv(output_matrix_path, index=False)

print(f"\n🎯 Golden Dataset Saved to: {output_matrix_path}")
print(f"📐 Final Correct Matrix Shape: {df_final_dataset.shape[0]} samples × {df_final_dataset.shape[1]} columns")

⏳ Processing 2836 genomes with Strict ACGT Filter...


Strict K-mer Extraction: 100%|█████████████████████████████████████████████████████| 2836/2836 [40:47<00:00,  1.16it/s]


📊 Building Feature Matrix...
🧹 Metadata cleaned. Rows reduced from 3064 to 3010 unique strains.
🔗 Performing strict merging...

🎯 Golden Dataset Saved to: ../data/processed/kmers_6_dataset.csv
📐 Final Correct Matrix Shape: 2836 samples × 4101 columns


In [4]:
import pandas as pd
import os

# 1. Load the newly generated golden dataset
output_matrix_path = "../data/processed/kmers_6_dataset.csv"

if os.path.exists(output_matrix_path):
    print("🧐 Running Automated Quality Control Check on Golden Dataset...\n")
    df_check = pd.read_csv(output_matrix_path)
    
    # 2. Shape Verification
    rows, cols = df_check.shape
    print(print(f"📐 Dataset Matrix Dimensions: {rows} rows (Strains) × {cols} columns (Features)"))
    
    # 3. Separate Metadata from K-mers
    metadata_cols = ['Genome ID', 'Genome Name', 'Antibiotic', 'Resistant Phenotype', 'AMR_Label']
    kmer_cols = [c for c in df_check.columns if c not in metadata_cols]
    
    print(f"ℹ️  Metadata Columns Count : {len(metadata_cols)}")
    print(f"🧬 Genomic K-mer Features  : {len(kmer_cols)}")
    
    # 4. Strict Pass/Fail Assertions
    print("\n🚨 --- Quality Control Gate Status ---")
    
    # Check 1: Exact K-mer Vocabulary Size
    if len(kmer_cols) == 4096:
        print("✅ PASS: K-mer vocabulary size is exactly 4,096 (4^6). Noise removed!")
    else:
        print(f"❌ FAIL: Expected 4096 k-mers, but got {len(kmer_cols)}. Check for ambiguous characters!")
        
    # Check 2: Row Inflation Check
    if rows <= 2835:
        print("✅ PASS: No row inflation detected. Metadata duplicates successfully purged.")
    else:
        print(f"⚠️  WARNING: Rows ({rows}) exceed unique physical FASTA files. Check merge constraints.")

    # Check 3: Check for remaining bad characters in column names
    bad_chars = [c for c in kmer_cols if any(char not in 'ATCG' for char in c)]
    if len(bad_chars) == 0:
        print("✅ PASS: 100% Strict ACGT compliance. No degenerate bases (M, S, W, R) found.")
    else:
        print(f"❌ FAIL: Found {len(bad_chars)} invalid columns containing non-ACGT letters: {bad_chars[:5]}")

    # 5. Class Balance Breakdown for Machine Learning Phase
    print("\n🎯 --- Target Label Distribution (AMR_Label) ---")
    balance = df_check['AMR_Label'].value_counts()
    percentages = df_check['AMR_Label'].value_counts(normalize=True) * 100
    
    for label, count in balance.items():
        label_name = "Resistant (1)" if label == 1 else "Susceptible (0)"
        print(f" 💊 {label_name:<15}: {count} samples ({percentages[label]:.2f}%)")

else:
    print(f"❌ File not found at: {output_matrix_path}. Please run the extraction pipeline first!")

🧐 Running Automated Quality Control Check on Golden Dataset...

📐 Dataset Matrix Dimensions: 2836 rows (Strains) × 4101 columns (Features)
None
ℹ️  Metadata Columns Count : 5
🧬 Genomic K-mer Features  : 4096

🚨 --- Quality Control Gate Status ---
✅ PASS: K-mer vocabulary size is exactly 4,096 (4^6). Noise removed!
⚠️  WARNING: Rows (2836) exceed unique physical FASTA files. Check merge constraints.
✅ PASS: 100% Strict ACGT compliance. No degenerate bases (M, S, W, R) found.

🎯 --- Target Label Distribution (AMR_Label) ---
 💊 Susceptible (0): 1764 samples (62.20%)
 💊 Resistant (1)  : 1072 samples (37.80%)
